# GraphGate: record refinement traces with an open-weight model

Runs the project's trace tools against an open-weight code model served inside this notebook, so no
commercial model is involved. Written for a **Kaggle** notebook with *Accelerator: GPU T4 x2* and
*Internet: On*; it also works on Colab with a T4.

Before you start, on your own machine run `python scripts/make_notebook_bundle.py` and add
`runs/graphgate_bundle.zip` to this notebook (Kaggle: **Add input > Upload**; Colab: the Files panel).

Run the cells top to bottom. A session has a time limit, so start with the pilot. Every model reply
is kept in a cache file: download the results at the end, and upload them again next session to
continue where you stopped.

In [ ]:
# Settings. The model is an Apache-2.0 code model; change it here only, and record what you used.
MODEL = "qwen2.5-coder:14b"
CONTEXT_TOKENS = 32768        # the largest slice plus a full rewrite must fit
BASE_URL = "http://localhost:11434/v1"
SEEDS = "0"                   # the pilot uses one replication; the dataset run uses "0,1,2"

In [ ]:
import glob, os, subprocess, zipfile

import shutil

bundle = (glob.glob("/kaggle/input/**/graphgate_bundle.zip", recursive=True)
          + glob.glob("/content/graphgate_bundle.zip") + glob.glob("graphgate_bundle.zip"))
# Kaggle may unpack an uploaded zip: then the project is a folder holding pyproject.toml.
unpacked = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
            if os.path.isdir(os.path.join(os.path.dirname(f), "src", "graphgate"))]
assert bundle or unpacked, "the bundle was not found: add graphgate_bundle.zip to the notebook first"
shutil.rmtree("/tmp/graphgate", ignore_errors=True)
if bundle:
    os.makedirs("/tmp/graphgate")
    zipfile.ZipFile(bundle[0]).extractall("/tmp/graphgate")
else:
    shutil.copytree(unpacked[0], "/tmp/graphgate")   # the input folder is read-only; work on a copy
print("project from", bundle[0] if bundle else unpacked[0])
os.chdir("/tmp/graphgate")

# Results of an earlier session, if you added them as an input: the cache makes finished turns free.
for earlier in glob.glob("/kaggle/input/**/graphgate_results.zip", recursive=True) + glob.glob("/content/graphgate_results.zip"):
    zipfile.ZipFile(earlier).extractall("/tmp/graphgate")
    print("restored", earlier)
# Kaggle may have unpacked that zip: then it is a folder holding runs/cache.sqlite.
for cache in glob.glob("/kaggle/input/**/runs/cache.sqlite", recursive=True):
    root = os.path.dirname(os.path.dirname(cache))
    for folder in ("runs", "data/traces"):
        if os.path.isdir(os.path.join(root, folder)):
            shutil.copytree(os.path.join(root, folder), os.path.join("/tmp/graphgate", folder), dirs_exist_ok=True)
    print("restored", root)

subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Install the model server and download the model (several GB; a few minutes).
import time

# The installer unpacks with zstd, which the notebook image does not ship.
subprocess.run("apt-get install -y -qq zstd", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
# Without this the server silently cuts every prompt to a few thousand tokens.
server_env = dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(CONTEXT_TOKENS), OLLAMA_KEEP_ALIVE="24h",
                  OLLAMA_NUM_PARALLEL="1")   # more than one slot splits the context between them
server = subprocess.Popen(["ollama", "serve"], env=server_env,
                          stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
subprocess.run(["ollama", "pull", MODEL], check=True)
# The context setting above is not always honoured, so it is also built into a
# named variant of the model, which every later cell uses.
BASE_MODEL = MODEL
MODEL = BASE_MODEL.replace(":", "-") + f"-ctx{CONTEXT_TOKENS}"
open("/tmp/Modelfile", "w").write(f"FROM {BASE_MODEL}\nPARAMETER num_ctx {CONTEXT_TOKENS}\n")
subprocess.run(["ollama", "create", MODEL, "-f", "/tmp/Modelfile"], check=True)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout,
      subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
# Check the context length took effect: a long prompt must be counted in full.
import sys

# The package was installed after this notebook started, so point Python at it directly.
sys.path.insert(0, "/tmp/graphgate/src")
from graphgate.config import ModelConfig
from graphgate.llm.openai_compat import OpenAICompatClient

client = OpenAICompatClient(ModelConfig.for_model(MODEL, max_tokens=64, provider="openai-compatible",
                                                  base_url=BASE_URL))
def lines(n):
    return "\n".join(f"line {i}: value = {i}" for i in range(n))

# Measure what a line costs in tokens, then send a prompt of about 24,000 tokens: more than
# half the window and less than all of it. A server that cut it short would count fewer.
small = client.complete("Answer in one word.", lines(300) + "\nWhat is the last line number?", replication=0)
per_line = small.usage["input_tokens"] / 300
count = int(24000 / per_line)
reply = client.complete("Answer in one word.", lines(count) + "\nWhat is the last line number?", replication=0)
print(f"{per_line:.1f} tokens per line; sent about {count * per_line:.0f}; server counted", reply.usage,
      "|", reply.text[:60], f"| {reply.latency_ms / 1000:.0f} s")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)
assert reply.usage["input_tokens"] > 20000, "the prompt was cut short: the context setting did not apply"


## Pilot: three events, one replication

About 21 model calls. Look at the timing before deciding on the full run.

In [ ]:
common = ["--provider", "openai-compatible", "--base-url", BASE_URL, "--model", MODEL, "--log-level", "INFO"]
subprocess.run(["python", "scripts/synthesize_traces.py", "--pilot", "--seeds", SEEDS, *common,
                "lollms-m45c", "nicegui-hxp3", "mako-2h4p"])
subprocess.run(["python", "scripts/label_traces.py", "--traces-dir", "runs/pilot-traces"])

In [ ]:
# What the pilot recorded, and how long the turns took.
import json

manifest = json.load(open("runs/pilot-traces/manifest.json"))
print("replications by outcome:", manifest["replications_by_outcome"])
for path in sorted(glob.glob("runs/pilot-traces/*.jsonl")):
    turns = [json.loads(line) for line in open(path) if '"kind": "turn"' in line]
    seconds = [t["latency_ms"] / 1000 for t in turns if t.get("latency_ms")]
    if seconds:
        print(f"{os.path.basename(path)}: {len(turns)} turns, {sum(seconds) / len(seconds):.0f} s per turn, "
              f"{sum(t['usage'].get('output_tokens', 0) for t in turns)} output tokens")

## Dataset run: the 30 validated events

Only after the pilot looks right. Up to 603 calls at three replications; it will not fit in one
session. Stop whenever you like: finished events are skipped and finished turns come from the cache
when you run this cell again, in this session or the next.

In [ ]:
# A background run is cut off at 12 hours and a run that is cut off keeps no output, so
# recording stops by itself well before that. Everything finished so far is then labelled
# and saved by the cells below, and the next run continues from it.
RECORDING_HOURS = 10.5
try:
    # Replications 0, 2 and 3. Under sampler seed 1 this model's first reply broke the
    # output format in every one of the first 14 events, so that seed is not used.
    subprocess.run(["python", "scripts/synthesize_traces.py", "--seeds", "0,2,3", *common],
                   timeout=RECORDING_HOURS * 3600)
except subprocess.TimeoutExpired:
    print(f"stopped recording after {RECORDING_HOURS} hours; run this notebook again to continue")
if glob.glob("data/traces/*.jsonl"):
    subprocess.run(["python", "scripts/label_traces.py"])
print(len(glob.glob("data/traces/*.jsonl")), "of 30 traces recorded")
print(len(glob.glob("data/traces/*.partial")), "more held back or unfinished (continued by the next run)")


## Save the results

Download `graphgate_results.zip` from the output panel and unpack it into the project folder on your
machine. Keep it: uploading it at the start of the next session continues the run.

In [ ]:
import shutil

os.makedirs("/tmp/out", exist_ok=True)
with zipfile.ZipFile("/tmp/out/graphgate_results.zip", "w", zipfile.ZIP_DEFLATED) as out:
    for folder in ("runs", "data/traces"):
        for path in glob.glob(folder + "/**", recursive=True):
            if os.path.isfile(path) and not path.endswith("graphgate_bundle.zip"):
                out.write(path)
    out.writestr("MODEL_USED.txt", MODEL + "\n" + subprocess.run(["ollama", "show", MODEL], capture_output=True, text=True).stdout)
target = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
shutil.copy("/tmp/out/graphgate_results.zip", target)
print("saved", os.path.join(target, "graphgate_results.zip"))